In [ ]:
import tkinter as tk
from tkinter import ttk

import numpy as np
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure


class RiemannSumApp:
    """Tkinter implementation of the Riemann sum calculator."""

    def __init__(self, root):
        self.root = root
        self.root.title("Riemann Sum Integration")
        self.root.minsize(800, 650)

        self.expression = tk.StringVar(value="x**2 + 3*x + 5")
        self.lower_bound = tk.StringVar(value="1")
        self.upper_bound = tk.StringVar(value="8")
        self.rectangle_count = tk.StringVar(value="10")
        self.method = tk.StringVar(value="Right")
        self.status = tk.StringVar(value="Enter values and click Compute.")

        controls = ttk.LabelFrame(root, text="Riemann Sum Parameters", padding=12)
        controls.pack(fill=tk.X, padx=12, pady=(12, 6))
        controls.columnconfigure(1, weight=1)

        ttk.Label(controls, text="f(x) =").grid(row=0, column=0, sticky=tk.W, padx=(0, 8), pady=4)
        ttk.Entry(controls, textvariable=self.expression, width=48).grid(
            row=0, column=1, columnspan=3, sticky=tk.EW, pady=4
        )

        ttk.Label(controls, text="Lower bound").grid(row=1, column=0, sticky=tk.W, padx=(0, 8), pady=4)
        ttk.Entry(controls, textvariable=self.lower_bound, width=12).grid(row=1, column=1, sticky=tk.W, pady=4)
        ttk.Label(controls, text="Upper bound").grid(row=1, column=2, sticky=tk.W, padx=(16, 8), pady=4)
        ttk.Entry(controls, textvariable=self.upper_bound, width=12).grid(row=1, column=3, sticky=tk.W, pady=4)

        ttk.Label(controls, text="N rectangles").grid(row=2, column=0, sticky=tk.W, padx=(0, 8), pady=4)
        ttk.Spinbox(controls, from_=1, to=200, textvariable=self.rectangle_count, width=10).grid(
            row=2, column=1, sticky=tk.W, pady=4
        )
        ttk.Label(controls, text="Method").grid(row=2, column=2, sticky=tk.W, padx=(16, 8), pady=4)
        ttk.Combobox(
            controls,
            textvariable=self.method,
            values=("Left", "Right", "Midpoint"),
            state="readonly",
            width=12,
        ).grid(row=2, column=3, sticky=tk.W, pady=4)

        ttk.Button(controls, text="Compute", command=self.compute).grid(
            row=3, column=0, columnspan=4, pady=(10, 2)
        )
        ttk.Label(controls, textvariable=self.status).grid(row=4, column=0, columnspan=4, sticky=tk.W, pady=(6, 0))

        self.figure = Figure(figsize=(7, 5), dpi=100)
        self.axes = self.figure.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.figure, master=root)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True, padx=12, pady=(6, 12))

    @staticmethod
    def make_function(expression):
        def function(x):
            return eval(expression, {"__builtins__": {}}, {"np": np, "x": x})

        return function

    def compute(self):
        try:
            lower_bound = float(self.lower_bound.get())
            upper_bound = float(self.upper_bound.get())
            rectangle_count = int(self.rectangle_count.get())
            if upper_bound <= lower_bound:
                raise ValueError("Upper bound must be greater than lower bound.")
            if not 1 <= rectangle_count <= 200:
                raise ValueError("N rectangles must be between 1 and 200.")

            function = self.make_function(self.expression.get())
            edges = np.linspace(lower_bound, upper_bound, rectangle_count + 1)
            widths = np.diff(edges)
            sample_points = {
                "Left": edges[:-1],
                "Right": edges[1:],
                "Midpoint": (edges[:-1] + edges[1:]) / 2,
            }[self.method.get()]
            heights = np.asarray(function(sample_points), dtype=float)
            if heights.shape == ():
                heights = np.full(rectangle_count, heights.item())
            if heights.shape != sample_points.shape or not np.all(np.isfinite(heights)):
                raise ValueError("f(x) must return finite numeric values for every sample point.")

            riemann_sum = np.sum(heights * widths)
            margin = (upper_bound - lower_bound) * 0.2
            x_values = np.linspace(lower_bound - margin, upper_bound + margin, 300)
            y_values = np.asarray(function(x_values), dtype=float)

            self.axes.clear()
            self.axes.plot(x_values, y_values, color="black", label="f(x)")
            for index in range(rectangle_count):
                self.axes.fill_between(
                    [edges[index], edges[index + 1]],
                    [heights[index], heights[index]],
                    0,
                    alpha=0.4,
                    color="tab:blue",
                    edgecolor="black",
                    linewidth=0.5,
                )
            self.axes.axhline(0, color="gray", linewidth=0.8)
            self.axes.set_xlabel("x")
            self.axes.set_ylabel("f(x)")
            self.axes.set_title(
                f"Riemann Sum ({self.method.get().lower()}) with N={rectangle_count}: "
                f"Approx. Integral = {riemann_sum:.4f}"
            )
            self.axes.legend()
            self.axes.grid(True, linestyle="--")
            self.figure.tight_layout()
            self.canvas.draw()
            self.status.set(f"Approximate integral (Riemann sum) = {riemann_sum:.6f}")
        except Exception as error:
            self.status.set(f"Error: {error}")


def main():
    root = tk.Tk()
    RiemannSumApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()